In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('movies_metadata.csv')

In [3]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [4]:
df.shape

(45466, 24)

In [5]:
df=df.drop_duplicates().reset_index(drop=True)

In [6]:
df=df[['title','overview','genres','tagline','vote_average','popularity']]

In [7]:
df.dropna(subset=['title'])

,title,overview,genres,tagline,vote_average,popularity
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",NaN,7.7,21.946943
1,Jumanji,When siblings Judy and Peter discover an encha...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Roll the dice and unleash the excitement!,6.9,17.015539
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",Friends are the people who let you be yourself...,6.1,3.859495
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,"[{'id': 35, 'name': 'Comedy'}]",Just When His World Is Back To Normal... He's ...,5.7,8.387519
...,...,...,...,...,...,...
45448,Subdue,Rising and falling between a man and woman.,"[{'id': 18, 'name': 'Drama'}, {'id': 10751, 'n...",Rising and falling between a man and woman,4.0,0.072051
45449,Century of Birthing,An artist struggles to finish his work while a...,"[{'id': 18, 'name': 'Drama'}]",NaN,9.0,0.178241
45450,Betrayal,"When one of her hits goes wrong, a professiona...","[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",A deadly game of wits.,3.8,0.903007
45451,Satan Triumphant,"In a small town live two brothers, one a minis...",[],NaN,0.0,0.003503


In [8]:
df['overview']=df['overview'].fillna('')

In [9]:
df.iloc[0].genres

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [10]:
import ast
df['genres']=df['genres'].apply(lambda x:" ".join([i['name'] for i in ast.literal_eval(x)]))

In [11]:
df['tagline']=df['tagline'].fillna('')

In [12]:
df['tags']=df['overview']+" "+df['genres']+" "+df['tagline']

In [13]:
df

,title,overview,genres,tagline,vote_average,popularity,tags
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,,7.7,21.946943,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,Roll the dice and unleash the excitement!,6.9,17.015539,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy,Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,Friends are the people who let you be yourself...,6.1,3.859495,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,Just When His World Is Back To Normal... He's ...,5.7,8.387519,Just when George Banks has recovered from his ...
...,...,...,...,...,...,...,...
45448,Subdue,Rising and falling between a man and woman.,Drama Family,Rising and falling between a man and woman,4.0,0.072051,Rising and falling between a man and woman. Dr...
45449,Century of Birthing,An artist struggles to finish his work while a...,Drama,,9.0,0.178241,An artist struggles to finish his work while a...
45450,Betrayal,"When one of her hits goes wrong, a professiona...",Action Drama Thriller,A deadly game of wits.,3.8,0.903007,"When one of her hits goes wrong, a professiona..."
45451,Satan Triumphant,"In a small town live two brothers, one a minis...",,,0.0,0.003503,"In a small town live two brothers, one a minis..."


In [14]:
df['tags'][1]

"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures. Adventure Fantasy Family Roll the dice and unleash the excitement!"

In [16]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\JASMINE\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\JASMINE\AppData\Roaming\nltk_data...


True

In [17]:
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()

In [18]:
def preprocess_text(text):
  text=str(text).lower() #lowercase
  text=re.sub(r'[^a-zA-Z]',' ',text)  #punctuation marks removed
  words=text.split()
  words=[word for word in words if word not in stop_words] #remove stop_words
  words=[lemmatizer.lemmatize(word) for word in words] #lemmatize
  return " ".join(words)

In [19]:
df['tags']=df['tags'].apply(preprocess_text)

In [20]:
df=df.reset_index(drop=True)

In [21]:
indices=pd.Series(df.index,index=df['title']).drop_duplicates()

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
tfidf=TfidfVectorizer(max_features=50000,ngram_range=(1,2),stop_words='english')
tfidf_matrix=tfidf.fit_transform(df['tags'])

In [24]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1586071 stored elements and shape (45453, 50000)>

In [25]:
def recommend(title, n=10):
  if title not in indices:
    return ['Movie not found!']
  idx=indices[title]
  sim_score=cosine_similarity(tfidf_matrix[idx],tfidf_matrix).flatten()
  similar_idx=sim_score.argsort()[::-1][1:n+1]
  return df['title'].iloc[similar_idx]

In [26]:
recommend('Dilwale')

36192                      Yaadein
33067                          Dor
36344                      Humraaz
34415    Har Dil Jo Pyar Karega...
35114                       Spirit
33873                     unINDIAN
36547                       Kismat
36193                       Indian
15016                Chalte Chalte
16429         Mujhse Dosti Karoge!
Name: title, dtype: object

In [27]:
import pickle

pickle.dump(tfidf_matrix,open('tfidf_matrix.pkl','wb'))
pickle.dump(indices,open('indices.pkl','wb'))
df.to_pickle('df.pkl')
pickle.dump(tfidf,open('tfidf.pkl','wb'))